# SIH26006 — Phase 8: Deep Learning (GRU & LSTM) Fixed Experiment

This self-contained notebook runs the **Phase 8 GRU and LSTM experiments** on Google Colab GPU with verified sequence alignment, target scaling, and metric logic.

### Verified Corrected Logic:
1. **Input(shape=...)**: Uses explicit Keras `Input` layer.
2. **Target Scaling**: Target $y$ is scaled (`StandardScaler` fit ONLY on train set) and inverse-transformed for evaluation.
3. **Directional Accuracy**: Compares predicted change ($\hat{y}_{t+h} - y_t$) vs actual change ($y_{t+h} - y_t$) using the **exact same unscaled baseline price $y_t$**.
4. **No-Lookahead Sequence Window**: 30-day sequence ends strictly at date $t$, while target is $y_{t+h}$.
5. **Exact Test Date Alignment**: 358 test dates matching Ridge & Persistence test set.

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 1: GPU DETECTION & REPOSITORY SETUP
# ─────────────────────────────────────────────────────────────
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

print('TensorFlow Version:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f'✅ GPU AVAILABLE: {len(gpus)} GPU device(s) detected.')
    for gpu in gpus:
        print(f'   Device Name: {gpu.name}')
        try:
            details = tf.config.experimental.get_device_details(gpu)
            if 'device_name' in details:
                print(f'   GPU Model: {details["device_name"]}')
        except Exception:
            pass
else:
    print('⚠️ WARNING: No GPU detected! Execution will fall back to CPU.')

# Clone repo and copy files to environment root if needed
if not os.path.exists('outputs/modeling_dataset.csv'):
    print('🌐 Auto-cloning repository from GitHub...')
    !git clone https://github.com/SSOHEB/FICOS-Platform.git repo_temp
    !cp -r repo_temp/* .
    !rm -rf repo_temp
    print('✅ Repository files copied to environment root.')

# Create output directories
os.makedirs('outputs/predictions/gru', exist_ok=True)
os.makedirs('outputs/predictions/lstm', exist_ok=True)
os.makedirs('outputs/plots/deep_learning', exist_ok=True)

if os.path.exists('outputs/modeling_dataset.csv'):
    print('✅ SUCCESS: outputs/modeling_dataset.csv is ready!')
else:
    print('❌ ERROR: outputs/modeling_dataset.csv missing.')

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 2: DATA PROCESSING & SEQUENCE CONSTRUCTION PIPELINE
# ─────────────────────────────────────────────────────────────
from sklearn.preprocessing import StandardScaler

# Set fixed random seeds
np.random.seed(42)
tf.random.set_seed(42)

df = pd.read_csv('outputs/modeling_dataset.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

# Chronological 70 / 15 / 15 Split
n = len(df)
n_train = int(n * 0.70)
n_val = int(n * 0.15)
n_test = n - n_train - n_val

print(f'Total Rows: {n} | Train: {n_train} | Val: {n_val} | Test: {n_test}')
print(f'Train Dates: {df["date"].iloc[0].strftime("%Y-%m-%d")} -> {df["date"].iloc[n_train-1].strftime("%Y-%m-%d")}')
print(f'Val Dates:   {df["date"].iloc[n_train].strftime("%Y-%m-%d")} -> {df["date"].iloc[n_train+n_val-1].strftime("%Y-%m-%d")}')
print(f'Test Dates:  {df["date"].iloc[n_train+n_val].strftime("%Y-%m-%d")} -> {df["date"].iloc[-1].strftime("%Y-%m-%d")}')

feature_cols = [c for c in df.columns if not c.startswith('target_') and c not in ['date']]
TARGETS = ['kdci', 'cape', 'panamax', 'supramax', 'handy']
HORIZONS = [1, 7, 14, 30]

def build_sequence_data(df, feature_cols, target_col, prev_col, n_train, n_val, lookback=30):
    '''Constructs 30-day lookback sequence data with strict train-only scaling and unscaled y_base.'''
    # Copy unscaled DataFrame for targets & base prices
    unscaled_df = df.copy()
    scaled_df = df.copy()
    
    # Impute missing feature values using train median ONLY
    train_medians = df.iloc[:n_train][feature_cols].median()
    scaled_df[feature_cols] = scaled_df[feature_cols].fillna(train_medians)
    
    # Feature Scaler (fit ONLY on train)
    scaler_X = StandardScaler()
    scaled_df.iloc[:n_train, scaled_df.columns.get_indexer(feature_cols)] = scaler_X.fit_transform(scaled_df.iloc[:n_train][feature_cols])
    scaled_df.iloc[n_train:, scaled_df.columns.get_indexer(feature_cols)] = scaler_X.transform(scaled_df.iloc[n_train:][feature_cols])
    
    # Target Scaler (fit ONLY on train target)
    scaler_y = StandardScaler()
    train_tgt_clean = unscaled_df.iloc[:n_train][target_col].dropna().values.reshape(-1, 1)
    scaler_y.fit(train_tgt_clean)
    
    # Helper to extract sequences for a slice
    def extract_slice(scaled_sub, unscaled_sub):
        X_l, y_raw_l, y_base_l, date_l = [], [], [], []
        feat_v = scaled_sub[feature_cols].values
        tgt_v = unscaled_sub[target_col].values
        base_v = unscaled_sub[prev_col].values # Raw unscaled price
        dt_v = unscaled_sub['date'].values
        
        for t in range(lookback - 1, len(scaled_sub)):
            if np.isnan(tgt_v[t]) or np.isnan(base_v[t]):
                continue
            seq = feat_v[t - lookback + 1 : t + 1, :]
            if np.isnan(seq).any():
                continue
            X_l.append(seq)
            y_raw_l.append(tgt_v[t])
            y_base_l.append(base_v[t])
            date_l.append(dt_v[t])
            
        return np.array(X_l), np.array(y_raw_l), np.array(y_base_l), np.array(date_l)
        
    # Train slice
    X_tr, y_tr_raw, _, _ = extract_slice(scaled_df.iloc[:n_train], unscaled_df.iloc[:n_train])
    # Val slice (include 30 lookback rows from end of train)
    val_start = max(0, n_train - lookback + 1)
    X_v, y_v_raw, _, _ = extract_slice(scaled_df.iloc[val_start:n_train+n_val], unscaled_df.iloc[val_start:n_train+n_val])
    # Test slice (include 30 lookback rows from end of val)
    test_start = max(0, n_train + n_val - lookback + 1)
    X_te, y_te_raw, y_base_te, dates_te = extract_slice(scaled_df.iloc[test_start:], unscaled_df.iloc[test_start:])
    
    # Scale y for neural network training
    y_tr_scaled = scaler_y.transform(y_tr_raw.reshape(-1, 1)).flatten()
    y_v_scaled = scaler_y.transform(y_v_raw.reshape(-1, 1)).flatten()
    
    return (X_tr, y_tr_scaled), (X_v, y_v_scaled), (X_te, y_te_raw, y_base_te, dates_te), scaler_y

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 3: KERAS GRU & LSTM MODEL BUILDERS (WITH EXPLICIT INPUT LAYER)
# ─────────────────────────────────────────────────────────────
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, GRU, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

def build_gru_model(input_shape):
    model = Sequential([
        Input(shape=input_shape),
        GRU(32, return_sequences=False),
        Dropout(0.2),
        Dense(16, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
    return model

def build_lstm_model(input_shape):
    model = Sequential([
        Input(shape=input_shape),
        LSTM(32, return_sequences=False),
        Dropout(0.2),
        Dense(16, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
    return model

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 4: METRIC EVALUATION & TRAINING LOOP
# ─────────────────────────────────────────────────────────────
def calc_metrics(y_true_raw, y_pred_raw, y_base_raw):
    '''
    Evaluates unscaled point MAE, RMSE, sMAPE, R-squared, and Directional Accuracy.
    Directional Accuracy compares predicted change (y_pred - y_base) vs actual change (y_true - y_base)
    relative to the EXACT SAME unscaled baseline price y_base at date t.
    '''
    mae = np.mean(np.abs(y_true_raw - y_pred_raw))
    rmse = np.sqrt(np.mean((y_true_raw - y_pred_raw)**2))
    smape = np.mean(200 * np.abs(y_pred_raw - y_true_raw) / (np.abs(y_true_raw) + np.abs(y_pred_raw) + 1e-8))
    ss_tot = np.sum((y_true_raw - np.mean(y_true_raw))**2)
    ss_res = np.sum((y_true_raw - y_pred_raw)**2)
    r2 = 1 - (ss_res / ss_tot) if ss_tot > 0 else np.nan
    
    actual_change = y_true_raw - y_base_raw
    pred_change = y_pred_raw - y_base_raw
    actual_dir = np.sign(actual_change)
    pred_dir = np.sign(pred_change)
    dir_acc = np.mean(actual_dir == pred_dir)
    
    return round(mae, 2), round(rmse, 2), round(smape, 2), round(r2, 4), round(dir_acc, 4)

deep_results = []
print('Starting Fixed Phase 8 Training Loop across 5 Targets x 4 Horizons...')

for tgt in TARGETS:
    for h in HORIZONS:
        target_col = f'target_{tgt}_{h}d'
        prev_col = tgt
        
        # Build sequences with unscaled y_base & target scaling
        (X_tr, y_tr_sc), (X_v, y_v_sc), (X_te, y_te_raw, y_base_te, dates_te), scaler_y = build_sequence_data(
            df, feature_cols, target_col, prev_col, n_train, n_val, lookback=30
        )
        
        if len(X_te) == 0:
            continue

        input_shape = (X_tr.shape[1], X_tr.shape[2])
        early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
        
        # 1. Train GRU
        gru = build_gru_model(input_shape)
        gru.fit(X_tr, y_tr_sc, validation_data=(X_v, y_v_sc), epochs=50, batch_size=32, callbacks=[early_stop], verbose=0)
        pred_g_sc = gru.predict(X_te, verbose=0)
        pred_g_raw = scaler_y.inverse_transform(pred_g_sc).flatten()
        
        mae_g, rmse_g, smape_g, r2_g, dacc_g = calc_metrics(y_te_raw, pred_g_raw, y_base_te)
        
        pd.DataFrame({'date': dates_te, 'y_true': y_te_raw, 'y_pred': pred_g_raw, 'y_prev': y_base_te}).to_csv(
            f'outputs/predictions/gru/{tgt}_{h}d.csv', index=False)
        
        deep_results.append({
            'model': 'GRU', 'freight_class': tgt, 'horizon': f'{h}d',
            'MAE': mae_g, 'RMSE': rmse_g, 'sMAPE': smape_g, 'R2': r2_g,
            'directional_accuracy': dacc_g, 'n_test': len(y_te_raw)
        })
        
        # 2. Train LSTM
        lstm = build_lstm_model(input_shape)
        lstm.fit(X_tr, y_tr_sc, validation_data=(X_v, y_v_sc), epochs=50, batch_size=32, callbacks=[early_stop], verbose=0)
        pred_l_sc = lstm.predict(X_te, verbose=0)
        pred_l_raw = scaler_y.inverse_transform(pred_l_sc).flatten()
        
        mae_l, rmse_l, smape_l, r2_l, dacc_l = calc_metrics(y_te_raw, pred_l_raw, y_base_te)
        
        pd.DataFrame({'date': dates_te, 'y_true': y_te_raw, 'y_pred': pred_l_raw, 'y_prev': y_base_te}).to_csv(
            f'outputs/predictions/lstm/{tgt}_{h}d.csv', index=False)
        
        deep_results.append({
            'model': 'LSTM', 'freight_class': tgt, 'horizon': f'{h}d',
            'MAE': mae_l, 'RMSE': rmse_l, 'sMAPE': smape_l, 'R2': r2_l,
            'directional_accuracy': dacc_l, 'n_test': len(y_te_raw)
        })
        
        print(f'  {tgt.upper():8s} {h:2d}d | GRU MAE: {mae_g:7.2f} (R2: {r2_g:6.4f}, Dir: {dacc_g:.1%}) | LSTM MAE: {mae_l:7.2f} (R2: {r2_l:6.4f}, Dir: {dacc_l:.1%})')

df_deep = pd.DataFrame(deep_results)
df_deep.to_csv('outputs/deep_model_comparison.csv', index=False)
print('\n✅ Saved Fixed Phase 8 results to outputs/deep_model_comparison.csv')

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 5: FULL 5-MODEL BENCHMARK MATRIX (PERSISTENCE vs RIDGE vs XGBOOST vs GRU vs LSTM)
# ─────────────────────────────────────────────────────────────
comp_paths = ['outputs/model_comparison.csv', 'FICOS-Platform/outputs/model_comparison.csv']
comp_file = next((p for p in comp_paths if os.path.exists(p)), None)

if comp_file:
    base_results = pd.read_csv(comp_file)
    full_comp = pd.concat([base_results, df_deep], ignore_index=True)
    full_comp.to_csv('outputs/full_5model_comparison.csv', index=False)
    print('✅ Saved 5-Model Comparison Matrix to outputs/full_5model_comparison.csv')
    
    print('\n' + '='*80)
    print('FULL MODEL COMPARISON (PERSISTENCE vs RIDGE vs XGBOOST vs GRU vs LSTM)')
    print('='*80)
    
    for tgt in TARGETS:
        print(f'\n  --- {tgt.upper()} ---')
        for h in HORIZONS:
            h_str = f'{h}d'
            sub = full_comp[(full_comp['freight_class'] == tgt) & (full_comp['horizon'] == h_str)]
            if sub.empty:
                continue
            best_row = sub.loc[sub['MAE'].idxmin()]
            pers_rows = sub[sub['model'] == 'Persistence']
            if not pers_rows.empty:
                pers_mae = pers_rows.iloc[0]['MAE']
                imp_pct = ((pers_mae - best_row['MAE']) / pers_mae) * 100
                print(f'    {h_str:3s}: Winner={best_row["model"]:11s} | MAE={best_row["MAE"]:7.2f} | R2={best_row["R2"]:6.4f} | DirAcc={best_row["directional_accuracy"]:.1%} | (vs Persistence: {imp_pct:+.1f}%)')
            else:
                print(f'    {h_str:3s}: Winner={best_row["model"]:11s} | MAE={best_row["MAE"]:7.2f} | R2={best_row["R2"]:6.4f} | DirAcc={best_row["directional_accuracy"]:.1%}')
else:
    print(df_deep.to_string())